In [4]:
# !pip install pymysql
# !pip install sqlalchemy

  Using cached sqlalchemy-2.0.50-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
  Using cached greenlet-3.5.1-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached sqlalchemy-2.0.50-cp314-cp314-win_amd64.whl (2.1 MB)
Using cached greenlet-3.5.1-cp314-cp314-win_amd64.whl (239 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

   ------------- -------------------------- 1/3 [greenlet]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ------------- 2/3 [sqlalchemy]
   -------------------------- ----

In [ ]:
import pymysql
import pandas as pd
from  sqlalchemy import create_engine

HOST='127.0.0.1'
PORT=3306
USER='scott'
PASS='tiger'
DB='python_schema'
TNAME = 'jdbct'

conn = pymysql.connect(host=HOST, port=PORT, database=DB, user=USER, password=PASS, charset='utf8mb4')
cursor = conn.cursor()
sql = f'select * from {TNAME} order by no'

In [10]:
print('(1) 방법 1 : cursor -> dataframe')
cursor.execute(sql)
rows = cursor.fetchall()
cols = [desc[0] for desc in cursor.description] # metadata
print(cols)          # ['no', 'name', 'rdate']
print(type(cols))    # list
df = pd.DataFrame(rows, columns=cols)
print(df)
print()
print(df.dtypes)
print()

(1) 방법 1 : cursor -> dataframe
['no', 'name', 'rdate']
<class 'list'>
   no name               rdate
0   1  이순신 2026-05-28 15:22:54
1   2  홍길동 2026-05-28 15:22:54
2   4  호날두 2026-05-28 16:23:13
3   5  김취추 2026-05-28 16:23:13
4   6  홍길순 2026-05-28 16:28:58
5   7  홍길자 2026-05-28 16:28:58

no                int64
name                str
rdate    datetime64[us]
dtype: object



In [16]:
print('(2) 방법 2 : sqlalchemy이용 ')
engine = create_engine(f'mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}/{DB}?charset=utf8mb4')
df2 = pd.read_sql(sql, engine)
print(df2.head())
print(type(df2))
cursor.close()
conn.close()

(2) 방법 2 : sqlalchemy이용 
   no name               rdate
0   1  이순신 2026-05-28 15:22:54
1   2  홍길동 2026-05-28 15:22:54
2   4  호날두 2026-05-28 16:23:13
3   5  김취추 2026-05-28 16:23:13
4   6  홍길순 2026-05-28 16:28:58
<class 'pandas.DataFrame'>


In [15]:
print('(3) dataframe 분석 ')

# df['name'].apply(len) : 각 행의 길이
# len(df['name']) : 전체 행의 개수


df['이름길이'] = df['name'].apply(len)
df.head()

(3) dataframe 분석 


,no,name,rdate,이름길이
0,1,이순신,2026-05-28 15:22:54,3
1,2,홍길동,2026-05-28 15:22:54,3
2,4,호날두,2026-05-28 16:23:13,3
3,5,김취추,2026-05-28 16:23:13,3
4,6,홍길순,2026-05-28 16:28:58,3


In [20]:
df[df['no'] >= 4]

,no,name,rdate,이름길이
2,4,호날두,2026-05-28 16:23:13,3
3,5,김취추,2026-05-28 16:23:13,3
4,6,홍길순,2026-05-28 16:28:58,3
5,7,홍길자,2026-05-28 16:28:58,3


In [24]:
print('(3) df -> DB 저장')
result_df = df[['no', 'name', '이름길이']].copy()
result_df_cols = ['no', 'name', 'name_len']

result_df.to_sql(
    name='jdbct_result', # 테이블명
    con = engine,
    if_exists='replace',
    index = False
)
print('저장완료')

(3) df -> DB 저장
저장완료


In [27]:
print('(4) 저장된 DB테이블에서 읽어오기')
db_result = pd.read_sql("select * from jdbct_result", engine)
db_result.head()

engine.dispose() # 엔진 연결해제

(4) 저장된 DB테이블에서 읽어오기


In [31]:
df = pd.read_csv('외교부_사건사고_전체데이터.csv')
df.head()


,continent_cd,continent_eng_nm,continent_nm,country_eng_nm,country_iso_alp2,country_nm,dang_map_download_url,flag_download_url,map_download_url,news,wrt_dt
0,50,Africa,아프리카,Ghana,GH,가나,https://www.0404.go.kr/files/download/FILE_000...,https://opendata.mofa.go.kr:8444/fileDownload/...,https://opendata.mofa.go.kr:8444/fileDownload/...,"<div>\r\n<h3 class=""tit"" style=""font-size: 18p...",2022-08-02
1,50,Africa,아프리카,Gabon,GA,가봉,https://www.0404.go.kr/files/download/COUNTRY_2/3,https://opendata.mofa.go.kr:8444/fileDownload/...,https://opendata.mofa.go.kr:8444/fileDownload/...,"<div>\r\n<h3 class=""tit"" style=""font-size: 18p...",2023-11-24
2,20,America,미주,Guyana,GY,가이아나,https://www.0404.go.kr/files/download/COUNTRY_...,https://opendata.mofa.go.kr:8444/fileDownload/...,https://opendata.mofa.go.kr:8444/fileDownload/...,"<div>\r\n<h3 class=""tit"" style=""font-size: 18p...",2024-08-01
3,50,Africa,아프리카,Gambia,GM,감비아,https://www.0404.go.kr/files/download/COUNTRY_5/3,https://opendata.mofa.go.kr:8444/fileDownload/...,https://opendata.mofa.go.kr:8444/fileDownload/...,"<div>\r\n<h3 class=""tit"" style=""font-size: 18p...",2024-06-28
4,20,America,미주,Guatemala,GT,과테말라,https://www.0404.go.kr/files/download/COUNTRY_7/3,https://opendata.mofa.go.kr:8444/fileDownload/...,https://opendata.mofa.go.kr:8444/fileDownload/...,"<div>\r\n<h3 class=""tit"" style=""font-size: 18p...",2022-08-02


In [34]:
df['continent_nm'].value_counts()

continent_nm
유럽      55
아프리카    48
아주      39
미주      36
중동      20
Name: count, dtype: int64

In [39]:
len(df['country_nm'].unique()) # 198개 나라 존재

198

In [ ]:
df['news'][0]

'<div>\r\n<h3 class="tit" style="font-size: 18px; font-weight: bold;">사건·사고 현황</h3>\r\n<p>\r\n[정정 상황 (전쟁, 내란, 테러 등)]<br>\r\nㅇ 일부지역에서 부족간 분쟁은 가끔 발생하고 있으나 전쟁 또는 내란의 발생징후는 없음. 그러나 연말연시 또는 선거철에는 치안이 불안해지는 경향이 있으므로 이 시기에는 야간 이동을 자제<br><br>\r\n\r\n[치안상황 (일반범죄 등)]<br>\r\nㅇ 다른 서부 아프리카에 비해 치안 상태가 좋은 편에 속하나 소매치기, 절도사건, 총기 강도 사건은 자주 발생하므로 주야간 모두 한적한 곳에 가는 것을 삼가야 함<br>\r\n</p>\r\n\r\n<br>\r\n\r\n<h3 class="tit" style="font-size: 18px; font-weight: bold;">사건·사고의 유형</h3>\r\n<p>\r\n※ 빈번한 사건·사고 유형 및 대처요령<br>\r\n\r\n[소매치기]<br>\r\nㅇ 주재국에서 소매치기 사건이 자주 발생하므로 중요한 물품은 가급적 휴대하지 말고 사람이 붐비는 곳에서 낯선 사람의 접근을 경계해야 함. 특히 공항에서 소매치기는 아니더라도 별도의 수고비를 요구하기 위해 여행객에게 접근하여 물건 등을 들어주거나 택시를 잡아주겠다며 호의를 베푸는 현지인을 주의<br><br>\r\n\r\n[차량 및 오토바이 강도]<br>\r\nㅇ 차량이 정차한 순간 문을 열고 소지품을 강탈 후 준비해 놓은 차를 타고 도주하는 사건이 발생하기도 하고, 운전 또는 정차 중인 차에 접근하여 강절도 행위가 발생하므로 항상 차량의 문을 잠그도록 하며 인적이 드문 곳에서의 주ㆍ정차를 삼가해야 함. 또한, 차내에 절도범의 표적이 될 만한 물건이 있을 경우 차량 유리를 파손 후 절도하는 경우가 있으므로 귀중품을 남겨두지 않도록 하며 부득이 남겨 놓을 경우 물건이 보이지 않도록 해야 함<br>\r\nㅇ 노상 또는 차량강도를 만났을 경우 우선 범인의 지

In [42]:
# !pip install bs4


   ------------- -------------------------- 1/3 [beautifulsoup4]
   ------------- -------------------------- 1/3 [beautifulsoup4]
   ---------------------------------------- 3/3 [bs4]



In [43]:
from bs4 import BeautifulSoup

def clean_html(text):
    if isinstance(text, str):
        return BeautifulSoup(text, 'html.parser').get_text(strip=True)
    return text

clean_html(df['news'][0])

'사건·사고 현황[정정 상황 (전쟁, 내란, 테러 등)]ㅇ 일부지역에서 부족간 분쟁은 가끔 발생하고 있으나 전쟁 또는 내란의 발생징후는 없음. 그러나 연말연시 또는 선거철에는 치안이 불안해지는 경향이 있으므로 이 시기에는 야간 이동을 자제[치안상황 (일반범죄 등)]ㅇ 다른 서부 아프리카에 비해 치안 상태가 좋은 편에 속하나 소매치기, 절도사건, 총기 강도 사건은 자주 발생하므로 주야간 모두 한적한 곳에 가는 것을 삼가야 함사건·사고의 유형※ 빈번한 사건·사고 유형 및 대처요령[소매치기]ㅇ 주재국에서 소매치기 사건이 자주 발생하므로 중요한 물품은 가급적 휴대하지 말고 사람이 붐비는 곳에서 낯선 사람의 접근을 경계해야 함. 특히 공항에서 소매치기는 아니더라도 별도의 수고비를 요구하기 위해 여행객에게 접근하여 물건 등을 들어주거나 택시를 잡아주겠다며 호의를 베푸는 현지인을 주의[차량 및 오토바이 강도]ㅇ 차량이 정차한 순간 문을 열고 소지품을 강탈 후 준비해 놓은 차를 타고 도주하는 사건이 발생하기도 하고, 운전 또는 정차 중인 차에 접근하여 강절도 행위가 발생하므로 항상 차량의 문을 잠그도록 하며 인적이 드문 곳에서의 주ㆍ정차를 삼가해야 함. 또한, 차내에 절도범의 표적이 될 만한 물건이 있을 경우 차량 유리를 파손 후 절도하는 경우가 있으므로 귀중품을 남겨두지 않도록 하며 부득이 남겨 놓을 경우 물건이 보이지 않도록 해야 함ㅇ 노상 또는 차량강도를 만났을 경우 우선 범인의 지시에 절대 순응하는 것이 좋으며, 금품을 지키기 위해 몸싸움을 할 경우 범인이 흉기를 사용하게 되어 생명의 위험까지 발생할 수 있으므로 절대 몸싸움을 해서는 안 됨. 또한, 소리를 질러서 도움을 요청할 때는 범인으로부터 안전거리를 확보한 상태에서 요청하는 것이 좋음[금융사기]ㅇ 로맨스 스캠 : 결혼을 빙자하여 SNS 등을 통해 항공료, 방문비용 등을 송금하도록 유도하니 각종 명목으로 금품을 요구하는 경우 유의ㅇ 거액 상속, 비자금 관련 사기: 거액의 상속금 또는 고위 공무원의 비자금이 은행에

In [44]:
clean_html(df['news'][1])

'사건ㆍ사고 현황[천재지변, 전쟁, 내란, 테러 등에 대한 상황 및 정세]ㅇ 가봉(Gabonese Republic)은 아프리카 중부 서해안, 대서양에 접하고 적도 바로 밑에 위치, 동쪽과 남쪽은 콩고 공화국, 북쪽은 카메룬 및 적도기니와 인접하고 있습니다.ㅇ 가봉은 1960년대 중반 이후 풍부한 천연자원(석유, 망간, 목재, 우라늄)에 힘입어 아프리카 국가 중 개인 소득 수준이 적도기니에 이어 두 번째로 높습니다.ㅇ 오마르 봉고 전 대통령의 장기 집권(41년) 이래 인근 지역 중 정세가 가장 안정된 편이나, 알리 봉고 현 대통령이 국외 요양 중 2019.1.7.(월) 수도 리브르빌에서 군사 쿠데타가 발생하여 수 시간 만에 진압되는 등, 현 정부에 대한 야당 및 국민들의 불만으로 치안 혼란 상황이 발생할 가능성은 있음[살인, 강도, 납치 등 범죄 피해 가능성 등 치안상태]ㅇ 인접국가와 특별한 분쟁은 없으며 테러 등 치안상태 역시 비교적 양호한 편이나 시내를 벗어난 현지인 밀집 거주 지역 출입은 자제가 요망됩니다.ㅇ 가난한 주변국으로부터 외국인 노동인력 유입에 대처하고 자국인 고용증가를 위하여 외국인 체류증 발급 제한과, 국경 검문검색 강화, 시내 차량 불심검문 등으로 외국인 유입을 제한하고 있습니다. 여행자는 여권, 운전자는 면허증을 반드시 휴대하도록 하고 검문시 제시하여야 합니다.ㅇ 외국인 노동자 장기 체류 요건이 미비한 경우 자국으로 추방하고 있으며, 이들 노동자 대부분 택시 운전사, 건설노동자, 가정부, 경비원 등으로 취업하여 근무하고 있습니다.ㅇ 외국인 불법체류자는 자국인에 비해 낮은 임금과 불리한 근로조건 등으로 장기적인 사회적 불안요인으로 작용하고 있으며, 이들을 통해 노상강도, 택시 강절도, 가택침입 절도, 권총강도 등의 범죄 사례가 발생하고 있습니다.ㅇ 기니만 해역에 해적에 의한 공격 및 피랍사건이 발생하고 있어 각별한 주의를 필요로 합니다.ㅇ 현재 코로나19 사태의 장기화와 그로 인한 불경기 등을 배경으로 아시아계에 대한 언어적·신체적 폭력 사건

In [45]:
clean_html(df['news'][2])

'사건ㆍ사고 현황ㅇ 주재국은 전쟁이나 내란 등은 없으며 또한 무장테러 단체 등도 없으나 극심한 빈부 부차, 높은 실업률 등으로 말미암아 사회치안이 매우 열악한 상황입니다.ㅇ 주재국은 치안불안 지역으로 분류되며, 시내공공장소에서 범죄가 국가적으로 주요 문제가 될 정도로 심각한 상황으로 도심지 범죄가 계속 증가추세에 있습니다.ㅇ 총기살해, 도로봉쇄, 반달리즘(파괴행위) 등 위범행위가 증가하고 있으며, 산발적이고 불시에 발생하는 경우가 많습니다.사건ㆍ사고의 유형ㅇ 수도인 Georgetown은 특히 주거침입, 유괴, 차량도난, 총기사고 등 폭력범죄가 많이 일어나며 범죄자들이 난폭해서 경찰조차도 피해자기 되기 쉬울 정도로 위험합니다.ㅇ 거리 범죄는 절도 및 강도상해(육체적인 폭력)가 일반적인 일이다. 특히 조지타운에서는 여행자가 어두워진 후에 걸어 다니는 것을 피해야 하며 항상 경계를 늦추지 않아야 하며 가능하면 혼자서 행동을 삼가야 합니다.ㅇ 길거리에서 환전을 피하는 것이 바람직합니다. 왜냐하면 위조지폐 가능성이 있으며 환전으로 현찰을 가지는 것이 발각될 시 범죄인들의 공격표적 대상이 될 수 있기 때문입니다.자연재해ㅇ 주재국은 카리브연안에 위치하지 않고 있어서 허리케인의 피해는 없는 편이며 지진 등 특수한 자연재해는 없는 편입니다. 그러나 가끔 폭우로 인해서 홍수피해는 있습니다.유의해야할 지역ㅇ Buxton(Georgtwon과 New Amsterdam 사이에 있는 지역)지역은 범죄조직의 근거지라고 알려져 있으므로 접근을 피할 것이 요망 됩니다. 또한 조지타운의 타이거 베이(Tiger Bay) 지역에서는 특히 조심해야 합니다.'

In [50]:
df2 = pd.read_csv('외교부_공관별 휴일현황_20220228.csv', encoding='cp949')
df3 = pd.read_csv('외교부_국가 지역별 재외공관 정보_20201231.csv' , encoding='cp949')
df4 = pd.read_csv('외교부_국가표준코드_20251222.csv')

df2.head()


,국가명,국가영문명,국제표준화기구(iso) 2자리코드,공관명,휴일명,휴일시작일,휴일종료일,휴일설명
0,네팔,Nepal,NP,주 네팔 대한민국 대사관,New Year's Day,2022-01-01,2022-01-01,NaN
1,네팔,Nepal,NP,주 네팔 대한민국 대사관,Makar Sankranti Festival,2022-01-15,2022-01-15,NaN
2,네팔,Nepal,NP,주 네팔 대한민국 대사관,Sonam Lhosar,2022-02-02,2022-02-02,NaN
3,네팔,Nepal,NP,주 네팔 대한민국 대사관,Shiva Ratri,2022-03-01,2022-03-01,NaN
4,네팔,Nepal,NP,주 네팔 대한민국 대사관,International Women's Day,2022-03-08,2022-03-08,NaN


In [57]:
df2['국가명'] = df2['국가명'].str.strip()
df2['휴일명'] = df2['휴일명'].str.strip()

In [58]:
df2[(df2.국가명=='네팔') & (df2.휴일명=="New Year's Day")]

,국가명,국가영문명,국제표준화기구(iso) 2자리코드,공관명,휴일명,휴일시작일,휴일종료일,휴일설명
0,네팔,Nepal,NP,주 네팔 대한민국 대사관,New Year's Day,2022-01-01,2022-01-01,NaN


In [59]:
df2[df2.휴일명=="New Year's Day"]

,국가명,국가영문명,국제표준화기구(iso) 2자리코드,공관명,휴일명,휴일시작일,휴일종료일,휴일설명
0,네팔,Nepal,NP,주 네팔 대한민국 대사관,New Year's Day,2022-01-01,2022-01-01,NaN
97,말레이시아,Malaysia,MY,주 말레이시아 대한민국 대사관,New Year's Day,2022-01-01,2022-01-01,NaN
282,파푸아뉴기니,Papua New Guinea :PNG,PG,주 파푸아뉴기니독립국 대한민국 대사관,New Year's Day,2022-01-01,2022-01-01,NaN
295,피지,Fiji,FJ,주 피지 대한민국 대사관,New Year's Day,2022-01-03,2022-01-03,NaN
321,캐나다,Canada,CA,주 캐나다 대한민국 대사관,New Year's Day,2022-01-01,2022-01-01,NaN
515,트리니다드토바고,Trinidad & Tobago,TT,주 트리니다드토바고 대한민국 대사관,New Year's Day,2022-01-01,2022-01-01,NaN
681,쿠웨이트,Kuwait,KW,주 쿠웨이트 대한민국 대사관,New Year's Day,2022-01-01,2022-01-01,NaN
1231,우간다,Uganda,UG,주 우간다 대한민국 대사관,New Year's Day,2022-01-01,2022-01-01,NaN
1249,짐바브웨,Zimbabwe,ZW,주 짐바브웨 대한민국 대사관,New Year's Day,2022-01-01,2022-01-01,NaN


In [65]:
df2['년도'] = df2['휴일시작일'].str.split('-').str[0]
df2['년도'].unique()

array(['2022'], dtype=object)

In [67]:
df4.head()

,국제표준화기구_2자리,국제표준화기구_3자리,국제표준화기구_숫자,대륙명_공통 대륙코드,대륙명_행정표준코드,대륙명_외교부 직제,영문명,한글명
0,GH,GHA,288.0,Africa,아프리카,아프리카,Ghana,가나
1,GA,GAB,266.0,Africa,아프리카,아프리카,Gabon,가봉
2,GY,GUY,328.0,America,남아메리카,미주,Guyana,가이아나
3,GM,GMB,270.0,Africa,아프리카,아프리카,Gambia,감비아
4,GG,GGY,831.0,Europe,유럽,유럽,Bailiwick of Guernsey,건지
